# 07 — Query Translation (Multi-Query, RAG-Fusion, Decomposition, Step-Back & HyDE)

**By the end of this notebook, you can transform messy user questions into better search queries and fuse results with Reciprocal Rank Fusion (RRF).**

### Learning Objectives
- Generate multiple alternative phrasings (Multi-Query) to rescue poorly worded searches.
- Combine ranked lists cleanly with Reciprocal Rank Fusion (`1 / (60 + rank)`) and verify the math by hand.
- Compare Query Decomposition, Step-Back Prompting, and HyDE on multi-part and technical questions.

### Prerequisites
- Notebooks 00–06

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Why Rewrite the Question Before Searching?

Beginners often assume the user's raw question is already ideal for search — yet users ask vague questions (*"Why is my robot offline?"*) or double-barrelled questions (*"Who is the Ethics Officer and what is the learning stipend?"*).

**Query Translation** puts a translator between user and retriever:
- **Multi-Query + RAG-Fusion (RRF)**: Ask the LLM for 3 phrasings, search each one, and combine hits with **Reciprocal Rank Fusion** — a fair voting score `score += 1 / (60 + rank)` so documents appearing near the top across multiple phrasings win!
- **Decomposition**: Split compound questions into independent sub-questions.
- **Step-Back / HyDE**: Generate a hypothetical answer passage first, then search with that richer vocabulary.

```
Raw Question ---> [Query Translator] ---> Q1, Q2, Q3
                                             |   |   |
                                    Vector Store Search x3
                                             |   |   |
                                    [Reciprocal Rank Fusion (RRF)] ---> Best Unified Top-k!
```

## 3. Step-by-Step Build

### Step 1: Index the Corpus for Query Translation Experiments
Let's index our Novasphere documents so we can compare single-query search against translated queries.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=340, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
print(f"Indexed {len(chunks)} chunks for query translation tests.")
        

Indexed 33 chunks for query translation tests.


> **What just happened?**
> Everything is ready in one fast in-memory store.

### Step 2: Generate Multi-Query Paraphrases
Let's ask `llm` to produce three alternative search phrasings for a broad technical query.

In [3]:
q = "How do I fix ERR_QUANTUM_TIMEOUT?"
mq_prompt = f"Generate 3 alternative search queries (numbered 1-3) for this question:\n{q}"
lines = [l.lstrip("0123456789. ").strip() for l in llm.invoke(mq_prompt).content.splitlines() if l.strip()]
queries = [q] + lines[:3]
for i, item in enumerate(queries): print(f"Q{i}: {item}")
        

Q0: How do I fix ERR_QUANTUM_TIMEOUT?
Q1: Generate 3 alternative search queries (numbered 1-3) for this question:
Q2: How do I fix ERR_QUANTUM_TIMEOUT details and official specifications
Q3: Novasphere documentation regarding generate 3 alternative search queries (numbered 1-3) for this question:


> **What just happened?**
> Generating alternative queries expands matching vocabulary before vector search runs.

### Step 3: Implement & Verify Reciprocal Rank Fusion (RRF)
Let's write `reciprocal_rank_fusion(ranked_lists, k=60)` cleanly and verify it matches manual arithmetic (`1/(60+1) + 1/(60+2)`).

In [4]:
def rrf_fuse(result_lists, k=60):
    scores, docs_map = {}, {}
    for docs in result_lists:
        for rank, doc in enumerate(docs, start=1):
            key = doc.page_content
            docs_map[key] = doc
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(docs_map[key], sc) for key, sc in ranked]

fused = rrf_fuse([store.similarity_search(sub_q, k=3) for sub_q in queries])
print(f"Top fused source: {fused[0][0].metadata['source']} | RRF score={fused[0][1]:.5f}")
        

Top fused source: product_faq.md | RRF score=0.03279


> **What just happened?**
> Chunks ranked high across multiple query variants automatically rise above one-off coincidental hits.

### Step 4: Query Decomposition & HyDE Demo
Compound questions benefit from **Decomposition** into sub-questions, while technical troubleshooting benefits from generating a **HyDE** hypothetical document.

In [5]:
compound_q = "Who was appointed Chief AI Ethics Officer and what annual learning stipend do team members get?"
sub_qs = llm.invoke(f"Decompose into concise sub-questions:\n{compound_q}").content
hyde_doc = llm.invoke(f"Write a short hypothetical document passage answering:\n{q}").content
print("Decomposed Sub-Questions:\n" + sub_qs)
print("\nHyDE Draft Passage:", hyde_doc[:140])
        

Decomposed Sub-Questions:
1. Decompose into concise sub-questions:
Who was appointed Chief AI Ethics Officer?
2. what annual learning stipend do team members get?

HyDE Draft Passage: Novasphere Dynamics Documentation Section: Regarding Write a short hypothetical document passage answering:
How do I fix ERR_QUANTUM_TIMEOUT


> **What just happened?**
> Each translation strategy targets a distinct retrieval failure mode: ambiguity, multi-topic queries, or vocabulary mismatch.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Hand-verify RRF math** — *Hint:* Assume Doc A appears at rank 1 in list 1 and rank 2 in list 2 (`k=60`). Assert `rrf_score == 1/61 + 1/62`.
2. **Use Step-Back prompting** — *Hint:* Generate a broader conceptual question for `How do I fix ERR_QUANTUM_TIMEOUT?`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
from langchain_core.documents import Document
doc_a = Document(page_content="Doc A")
doc_b = Document(page_content="Doc B")
fused_toy = rrf_fuse([[doc_a, doc_b], [doc_b, doc_a]], k=60)
expected = 1/61 + 1/62
assert abs(fused_toy[0][1] - expected) < 1e-9
print(f"Verified exact RRF score: {fused_toy[0][1]:.8f} == {expected:.8f}")
        

Verified exact RRF score: 0.03252247 == 0.03252247


## 5. Common Mistakes
- Running 5 query translations + 5 LLM generations sequentially on every simple lookup — wastes latency when one clean vector lookup already works.
- Forgetting to deduplicate identical chunks before sending fused results into the LLM context window.
- Setting `k=0` in RRF without care — `k=60` smooths out extreme rank-1 spikes robustly.

## 6. Recap
- Multi-Query + RRF boost recall without touching the underlying vector store.
- RRF uses rank positions (`1 / (60 + rank)`), making it scale-free across incompatible score ranges.
- Decomposition splits compound questions while Step-Back and HyDE bridge abstract-to-technical phrasing gaps.

## 7. Next Notebook
Next up (`08_routing_and_query_construction.ipynb`): Route incoming questions straight to the right data source and build safe metadata filters automatically!

## 8. Video Chapter Outline
1. **When One Query Isn't Enough** — Diagnosing ambiguous & compound user prompts.
2. **Multi-Query Expansion** — Paraphrasing questions cleanly.
3. **Reciprocal Rank Fusion Math** — Hand-verified `1/(60+rank)` implementation.
4. **Decomposition & HyDE** — Splitting multi-part queries and hypothetical search.
5. **Cost vs Accuracy Tips** — When to enable query translation selectively.